7. In Sections 5.1.2 and 5.1.3, we saw that the cross_validate() function
can be used in order to compute the LOOCV test error estimate.
Alternatively, one could compute those quantities using just sm.GLM()
and the predict() method of the fitted model within a for loop. You
will now take this approach in order to compute the LOOCV error
for a simple logistic regression model on the Weekly data set. Recall
that in the context of classification problems, the LOOCV error is
given in (5.4).

(a) Fit a logistic regression model that predicts Direction using Lag1
and Lag2.

In [18]:
from ISLP import load_data
from ISLP.models import(ModelSpec as MS, summarize)
import statsmodels.api as sm
from sklearn.model_selection import cross_validate
import pandas as pd
import numpy as np

In [19]:
Weekly = load_data('Weekly')
Weekly.head(3)

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up


In [20]:
X = MS(['Lag1','Lag2']).fit_transform(Weekly)
y = Weekly['Direction'] == 'Up'

resultados = sm.GLM(y, X, family=sm.families.Binomial()).fit()
print(resultados.bse)
summarize(resultados)

intercept    0.061466
Lag1         0.026217
Lag2         0.026546
dtype: float64


,coef,std err,z,P>|z|
intercept,0.2212,0.061,3.599,0.000
Lag1,-0.0387,0.026,-1.477,0.140
Lag2,0.0602,0.027,2.270,0.023


(b) Fit a logistic regression model that predicts Direction using Lag1
and Lag2 using all but the first observation.

In [21]:
X = MS(['Lag1','Lag2']).fit_transform(Weekly)
X_treino, X_teste = X.iloc[1:], X.iloc[0]
y_treino, y_teste = y.iloc[1:], y.iloc[0]

resultados = sm.GLM(y_treino, X_treino, family=sm.families.Binomial()).fit()
print(resultados.bse)
summarize(resultados)


intercept    0.061499
Lag1         0.026219
Lag2         0.026561
dtype: float64


,coef,std err,z,P>|z|
intercept,0.2232,0.061,3.630,0.000
Lag1,-0.0384,0.026,-1.466,0.143
Lag2,0.0608,0.027,2.291,0.022


(c) Use the model from (b) to predict the direction of the first obser
vation. You can do this by predicting that the first observation
will go up if P(Direction = "Up"|Lag1, Lag2) > 0.5. Was this
observation correctly classified?

In [22]:
predicao = resultados.predict(X_teste) > 0.5
predicao == y_teste


None    False
dtype: bool

(d) Write a for loop from i =1to i = n, where n is the number of
observations in the data set, that performs each of the following
steps:

i. Fit a logistic regression model using all but the ith obser
vation to predict Direction using Lag1 and Lag2.

ii. Compute the posterior probability of the market moving up
for the ith observation.

iii. Use the posterior probability for the ith observation in order
to predict whether or not the market moves up.

iv. Determine whether or not an error was made in predicting
the direction for the ith observation. If an error was made,
then indicate this as a 1, and otherwise indicate it as a 0.

In [23]:
erros = np.zeros(Weekly.shape[0])
for i in range(Weekly.shape[0]):
    X = MS(['Lag1','Lag2']).fit_transform(Weekly)
    X_treino, X_teste = X.drop(index=i), X.iloc[i]
    y_treino, y_teste = y.drop(index=i), y.iloc[i]
    resultados = sm.GLM(y_treino, X_treino, family=sm.families.Binomial()).fit()
    if ((resultados.predict(X_teste)>0.5) != y_teste).all():
        erros[i] = 1

In [24]:
np.average(erros)

np.float64(0.44995408631772266)

O modelo teve uma taxa de erro de 44.9%, que para um modelo de ações é um resultado razoável